# Milestone 2: Data Quality dan Profiling

Lanjutkan dari hasil `01_data_profiling.ipynb`. Notebook ini menerapkan profiling DuckDB `SUMMARIZE` dan transformasi Polars untuk menangani missing values, outlier, duplikat, serta validasi data dasar. Sesuaikan nama kolom dan aturan validasi dengan dataset masing-masing.

In [5]:
from pathlib import Path

import duckdb
import polars as pl

PROJECT_ROOT = Path.cwd().resolve().parent
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "tracks_features.parquet"
OUTPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'tracks_features.parquet'

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f'Dataset tidak ditemukan: {DATA_PATH}. '
        'Pastikan Milestone 1 sudah menempatkan data di data/raw/.'
    )

df = pl.read_parquet(DATA_PATH)
print(f'Ukuran awal: {df.shape[0]:,} baris x {df.shape[1]} kolom')
df.head()

Ukuran awal: 1,204,025 baris x 24 kolom


id,name,album,album_id,artists,artist_ids,track_number,disc_number,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms,time_signature,year,release_date
str,str,str,str,str,str,i64,i64,bool,f64,f64,i64,f64,i64,f64,f64,f64,f64,f64,f64,i64,f64,i64,str
"""7lmeHLHBe4nmXzuXc0HDjk""","""Testify""","""The Battle Of Los Angeles""","""2eia0myWFgoHuttJytCxgX""","""['Rage Against The Machine']""","""['2d0hyoQ5ynDBnkvAbJKORj']""",1,1,false,0.47,0.978,7,-5.399,1,0.0727,0.0261,0.0000109,0.356,0.503,117.906,210133,4.0,1999,"""1999-11-02"""
"""1wsRitfRRtWyEapl0q22o8""","""Guerrilla Radio""","""The Battle Of Los Angeles""","""2eia0myWFgoHuttJytCxgX""","""['Rage Against The Machine']""","""['2d0hyoQ5ynDBnkvAbJKORj']""",2,1,true,0.599,0.957,11,-5.764,1,0.188,0.0129,0.0000706,0.155,0.489,103.68,206200,4.0,1999,"""1999-11-02"""
"""1hR0fIFK2qRG3f3RF70pb7""","""Calm Like a Bomb""","""The Battle Of Los Angeles""","""2eia0myWFgoHuttJytCxgX""","""['Rage Against The Machine']""","""['2d0hyoQ5ynDBnkvAbJKORj']""",3,1,false,0.315,0.97,7,-5.424,1,0.483,0.0234,0.000002,0.122,0.37,149.749,298893,4.0,1999,"""1999-11-02"""
"""2lbASgTSoDO7MTuLAXlTW0""","""Mic Check""","""The Battle Of Los Angeles""","""2eia0myWFgoHuttJytCxgX""","""['Rage Against The Machine']""","""['2d0hyoQ5ynDBnkvAbJKORj']""",4,1,true,0.44,0.967,11,-5.83,0,0.237,0.163,0.000004,0.121,0.574,96.752,213640,4.0,1999,"""1999-11-02"""
"""1MQTmpYOZ6fcMQc56Hdo7T""","""Sleep Now In the Fire""","""The Battle Of Los Angeles""","""2eia0myWFgoHuttJytCxgX""","""['Rage Against The Machine']""","""['2d0hyoQ5ynDBnkvAbJKORj']""",5,1,false,0.426,0.929,2,-6.729,1,0.0701,0.00162,0.105,0.0789,0.539,127.059,205600,4.0,1999,"""1999-11-02"""


## 1. Profiling dengan DuckDB `SUMMARIZE`

Dapatkan ringkasan statistik seluruh kolom (tipe data, null%, min/max, distinct count) dalam satu perintah.

In [6]:
connection = duckdb.connect()
connection.register('dataset', df)

profil = connection.execute('SUMMARIZE SELECT * FROM dataset').pl()
profil

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,"decimal[9,2]"
"""id""","""VARCHAR""","""0001Lyv0YTjkZSqzT4WkLy""","""7zzzxEQAsiuIimY7AiTGJ9""",1118017,null,null,null,null,null,1204025,0.00
"""name""","""VARCHAR""","""!""","""﻿what Baby""",1064273,null,null,null,null,null,1204025,0.00
"""album""","""VARCHAR""","""!""","""행복(Happiness)""",129733,null,null,null,null,null,1204025,0.00
"""album_id""","""VARCHAR""","""0008WZMLnvEBVnq418uZsI""","""7zzz0Fqahwtd6jd2BgBFj6""",103327,null,null,null,null,null,1204025,0.00
"""artists""","""VARCHAR""","""[""'63 Burnout""]""","""['Ｏｃｅａｎ Ｓｈｏｒｅｓ']""",147447,null,null,null,null,null,1204025,0.00
…,…,…,…,…,…,…,…,…,…,…,…
"""tempo""","""DOUBLE""","""0.0""","""248.93400000000003""",141624,"""117.63435180997067""","""30.937046661114245""","""94.08907724876458""","""116.70046515610359""","""137.03102496551085""",1204025,0.00
"""duration_ms""","""BIGINT""","""1000""","""6061090""",217629,"""248839.86096218933""","""162210.357230304""","""173956""","""224339""","""285891""",1204025,0.00
"""time_signature""","""DOUBLE""","""0.0""","""5.0""",5,"""3.8324943418948942""","""0.5611825532708646""","""4.0""","""4.0""","""4.0""",1204025,0.00


## 2. Deteksi Missing Values

Lihat jumlah dan persentase null per kolom sebelum memutuskan strategi penanganan.

In [7]:
jumlah_baris = len(df)
null_summary = (
    df.null_count()
    .transpose(include_header=True, header_name='kolom', column_names=['jumlah_null'])
    .with_columns(
        (pl.col('jumlah_null') / jumlah_baris * 100).round(2).alias('persen_null')
    )
    .filter(pl.col('jumlah_null') > 0)
    .sort('jumlah_null', descending=True)
)
null_summary

kolom,jumlah_null,persen_null
str,u32,f64


## 3. Penanganan Missing Values

Ganti `kolom_numerik` dan `kolom_kategorik` dengan nama kolom asli pada dataset. Pilih strategi sesuai jenis data: median/mean untuk numerik, nilai default atau modus untuk kategorik.

In [8]:
# TODO: sesuaikan nama kolom dengan dataset Anda
df_missing_handled = df.with_columns([
    # Contoh: isi kolom numerik dengan median
    # pl.col('kolom_numerik').fill_null(pl.col('kolom_numerik').median()),

    # Contoh: isi kolom kategorik dengan nilai default
    # pl.col('kolom_kategorik').fill_null('tidak diketahui'),
])

print(f'Null sebelum: {df.null_count().sum_horizontal().sum():,}')
print(f'Null sesudah: {df_missing_handled.null_count().sum_horizontal().sum():,}')

Null sebelum: 0
Null sesudah: 0


## 4. Deteksi dan Penanganan Outlier

Gunakan metode IQR sebagai titik awal, lalu kombinasikan dengan batas bisnis (domain knowledge) yang relevan dengan dataset Anda.

In [16]:
# Memilih semua kolom numerik
KOLOM_NUMERIK = [
    col for col, dtype in df_missing_handled.schema.items()
    if dtype.is_numeric()
]

print("Kolom numerik:", KOLOM_NUMERIK)

# Menyimpan batas IQR setiap kolom
batas_iqr = {}

for col in KOLOM_NUMERIK:
    q1 = df_missing_handled[col].quantile(0.25)
    q3 = df_missing_handled[col].quantile(0.75)
    iqr = q3 - q1

    batas_bawah = q1 - 1.5 * iqr
    batas_atas = q3 + 1.5 * iqr

    batas_iqr[col] = (batas_bawah, batas_atas)

    jumlah_outlier = df_missing_handled.select(
        (
            (pl.col(col) < batas_bawah)
            | (pl.col(col) > batas_atas)
        ).sum()
    ).item()

    print(
        f"{col}: Q1={q1}, Q3={q3}, IQR={iqr}, "
        f"outlier={jumlah_outlier:,}"
    )

# Winsorization pada seluruh kolom numerik
df_outlier_handled = df_missing_handled.with_columns([
    pl.col(col).clip(
        lower_bound=lower,
        upper_bound=upper
    ).alias(col)
    for col, (lower, upper) in batas_iqr.items()
])

print("\nPenanganan outlier selesai.")
print(f"Ukuran data: {df_outlier_handled.shape}")

Kolom numerik: ['track_number', 'disc_number', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms', 'time_signature', 'year']
track_number: Q1=3.0, Q3=10.0, IQR=7.0, outlier=42,957
disc_number: Q1=1.0, Q3=1.0, IQR=0.0, outlier=53,764
danceability: Q1=0.35600000000000004, Q3=0.633, IQR=0.27699999999999997, outlier=0
energy: Q1=0.252, Q3=0.7659999999999999, IQR=0.5139999999999999, outlier=0
key: Q1=2.0, Q3=8.0, IQR=6.0, outlier=0
loudness: Q1=-15.254000000000001, Q3=-6.7170000000000005, IQR=8.537, outlier=38,727
mode: Q1=0.0, Q3=1.0, IQR=1.0, outlier=0
speechiness: Q1=0.0351, Q3=0.0723, IQR=0.037200000000000004, outlier=167,338
acousticness: Q1=0.0376, Q3=0.861, IQR=0.8234, outlier=0
instrumentalness: Q1=7.6e-06, Q3=0.7190000000000001, IQR=0.7189924000000001, outlier=0
liveness: Q1=0.0968, Q3=0.245, IQR=0.1482, outlier=95,938
valence: Q1=0.191, Q3=0.644, IQR=0.453, outlier=0
tempo: Q1=94.054

## 5. Deduplikasi

Tentukan kolom kunci bisnis (misalnya ID unik) sebelum menghapus duplikat — jangan deduplikasi berdasarkan seluruh baris.

In [12]:
# TODO: ganti 'id_unik' dengan kolom kunci unik pada dataset Anda
KOLOM_KUNCI = 'id'

total_baris = len(df_outlier_handled)
baris_unik = df_outlier_handled[KOLOM_KUNCI].n_unique() if KOLOM_KUNCI in df_outlier_handled.columns else total_baris
print(f'Total baris: {total_baris:,}')
print(f'Baris unik berdasarkan {KOLOM_KUNCI}: {baris_unik:,}')
print(f'Duplikat: {total_baris - baris_unik:,}')

df_dedup = df_outlier_handled.unique(
    subset=[KOLOM_KUNCI] if KOLOM_KUNCI in df_outlier_handled.columns else None,
    keep='first',
    maintain_order=True,
)

Total baris: 1,204,025
Baris unik berdasarkan id: 1,204,025
Duplikat: 0


## 6. Validasi Data Dasar

Tulis beberapa aturan validasi sederhana untuk memastikan data bersih memenuhi batas yang masuk akal. Gantikan contoh aturan di bawah dengan aturan yang relevan dengan dataset Anda.

In [14]:
# Hubungkan DataFrame hasil deduplikasi ke DuckDB
connection.register("dataset_bersih", df_dedup)

# Validasi dasar dataset Spotify
validasi = connection.execute("""
    SELECT
        COUNT(*) AS total_baris,

        SUM(CASE
            WHEN id IS NULL OR TRIM(id) = '' THEN 1
            ELSE 0
        END) AS id_kosong,

        SUM(CASE
            WHEN duration_ms IS NULL OR duration_ms <= 0 THEN 1
            ELSE 0
        END) AS durasi_tidak_valid,

        SUM(CASE
            WHEN danceability < 0 OR danceability > 1
              OR energy < 0 OR energy > 1
              OR valence < 0 OR valence > 1
            THEN 1
            ELSE 0
        END) AS atribut_audio_di_luar_rentang,

        SUM(CASE
            WHEN year IS NOT NULL AND (year < 1900 OR year > 2026)
            THEN 1
            ELSE 0
        END) AS tahun_tidak_wajar,

        SUM(CASE
            WHEN tempo IS NOT NULL AND tempo <= 0
            THEN 1
            ELSE 0
        END) AS tempo_tidak_valid

    FROM dataset_bersih
""").pl()

validasi

total_baris,id_kosong,durasi_tidak_valid,atribut_audio_di_luar_rentang,tahun_tidak_wajar,tempo_tidak_valid
i64,"decimal[38,0]","decimal[38,0]","decimal[38,0]","decimal[38,0]","decimal[38,0]"
1204025,0,0,0,0,0


## 7. Simpan Data Bersih

Simpan hasil ke `data/processed/` agar dapat dipakai pada Milestone 3 tanpa mengulang proses cleaning.

In [15]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_dedup.write_parquet(OUTPUT_PATH)

print(f'Baris sebelum cleaning: {len(df):,}')
print(f'Baris sesudah cleaning: {len(df_dedup):,}')
print(f'Data bersih tersimpan di: {OUTPUT_PATH}')

Baris sebelum cleaning: 1,204,025
Baris sesudah cleaning: 1,204,025
Data bersih tersimpan di: /home/jovyan/work/data/processed/tracks_features.parquet


## Catatan Milestone 2

Dokumentasikan di sini: dimensi kualitas data yang paling bermasalah, strategi yang dipilih untuk missing values dan outlier, jumlah baris yang berubah, serta aturan validasi yang diterapkan.

Disini tidak ada dimensi data yang bermasalah dan tidak ada missing values juga